# Retrieval descriptor diagnostic — recall vs. window, top-1 vs. window

Supports the corrected narrative for `sec:rd-ri`: a candidate filter's usefulness
depends on whether its prediction error is small relative to its own dynamic
range, not on the descriptor's held-out accuracy in isolation. RI and
heavy-atom count (HA) both have good held-out accuracy (AIRI R^2=0.977 StdNP;
HA predictor R^2=0.75) but fail as filters because their prediction error is
large relative to their dynamic range, so windows narrow enough to shrink the
pool exclude the true structure too often (low recall). MW needs no separate
prediction model (read directly from the query's own predicted spectrum) and
is the only descriptor that beats unfiltered retrieval.

**Main-text panels are all restricted to the single StdNP-having query
subset (n=1,168)**, so RI-alone, MW-alone, HA-alone, and the RI-union
tracks are directly comparable on the same queries -- no population
mismatch. The full 27,647-query (no-RI-restriction) version of the
MW/HA-alone panels, matching Table 1 / `tab:pubchem-global-nofilter` and
`tab:si-recall-vs-pool`, is generated separately at the end for the SI.

ICICLE, cosine similarity, autofail scoring.
Data: `results/pubchem_retrieval_eval_icicle_rerun_260710/`.

### Caveat: the StdNP subset is a systematically easier query population

The StdNP-having subset (n=1,168) is NOT a random sample of the full test
set (n=27,647) -- it is systematically biased toward smaller, simpler
molecules (standard non-polar GC columns are the classic column type for
small, volatile analytes; larger/more polar compounds are less often run
on them or don't elute cleanly, so they're under-represented in StdNP
labels). Concretely (`data/NIST2023_GCMS_main/metadata.tsv`):

| | Full test set (n=27,647) | StdNP subset (n=1,168) |
|---|---|---|
| Median MW (Da) | 281 | 197.5 (30% lower) |
| Median heavy-atom count | 19 | 13 (32% fewer) |

Smaller molecules have fewer plausible PubChem near-isomers to confuse
retrieval with, and a fixed-width MW/heavy-atom window covers a
proportionally larger share of a narrower population. Both effects inflate
absolute MW/HA top-1 on the StdNP subset relative to the full test set
(e.g. MW ±80 Da: 13.4% here vs. 12.1% on the full set -- Table
`tab:si-mw-only`). **The StdNP-subset panels below are the only ones that
can show the RI-union tracks (union requires an RI value), but their
absolute MW/HA numbers should not be read as "how well MW/HA perform in
general" -- use the full-test-set SI panel at the end of this notebook for
that.**

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from icicle.utils.visualization.style import (
    get_palette,
    make_fig,
    save_fig,
    set_style,
)

set_style("manuscript")
palette = get_palette()

RESULTS_DIR = Path("../../results/pubchem_retrieval_eval_icicle_rerun_260710")
OUTPUT_DIR = Path("../../figures/retrieval_descriptor_diagnostic")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LIBRARY_SIZE = 93_661_074

## Load per-query data — StdNP subset (n=1,168) only

MW/HA/RI here all use the `*_StdNP.tsv` per-query files, restricted to the
same n=1,168 StdNP-having queries. Each has `n_candidates` (pool size) and
`true_mol_found` (recall indicator) plus `rank_autofail_cosine` (for top-1).

In [ ]:
MW_FILES = {
    "\u00b180 Da": RESULTS_DIR / "retrieval_per_query_mw80_StdNP.tsv",
    "\u00b110 Da": RESULTS_DIR / "retrieval_per_query_mw10_StdNP.tsv",
    "\u00b15 Da": RESULTS_DIR / "retrieval_per_query_mw5_StdNP.tsv",
}

HA_FILES = {
    "\u00b11": RESULTS_DIR / "retrieval_per_query_heavy_atom1_StdNP.tsv",
    "\u00b12": RESULTS_DIR / "retrieval_per_query_heavy_atom2_StdNP.tsv",
    "\u00b13": RESULTS_DIR / "retrieval_per_query_heavy_atom3_StdNP.tsv",
    "\u00b16": RESULTS_DIR / "retrieval_per_query_heavy_atom6_StdNP.tsv",
    "\u00b18": RESULTS_DIR / "retrieval_per_query_heavy_atom8_StdNP.tsv",
}

RI_POOL_SIZES = [1_000, 100_000, 1_000_000, 10_000_000]

mw_dfs = {label: pd.read_csv(f, sep="\t") for label, f in MW_FILES.items()}
ha_dfs = {label: pd.read_csv(f, sep="\t") for label, f in HA_FILES.items()}
ri_dfs = {
    n: pd.read_csv(
        RESULTS_DIR / f"retrieval_per_query_StdNP_{n}.tsv", sep="\t"
    )
    for n in RI_POOL_SIZES
}

with open(RESULTS_DIR / "retrieval_ablation_StdNP.json") as fh:
    ri_ladder = json.load(fh)
unfiltered_top1_stdnp = ri_ladder["all"]["autofail"]["cosine"][
    "top_1_accuracy"
]

print("MW n per width:", {k: len(v) for k, v in mw_dfs.items()})
print("HA n per width:", {k: len(v) for k, v in ha_dfs.items()})
print("RI n per pool size:", {k: len(v) for k, v in ri_dfs.items()})
print("Unfiltered top-1 on StdNP subset:", unfiltered_top1_stdnp)

## Load union per-query data (StdNP subset, N=1,000)

Requires the per-query dump added to `evaluate_ri_mw_union` /
`evaluate_ri_heavy_atom_union` in this session (previously these functions
only wrote aggregate top-k/MRR, no per-query `n_candidates`/`true_mol_found`).
MW-union rerun (N=1,000 only) is in progress as of writing; HA-union has no
per-query dump yet (background jobs launched before the patch). Cells below
load whichever files exist and print which are still missing -- no numbers
are fabricated for missing files.

In [ ]:
MW_UNION_FILES = {
    "\u00b180 Da": RESULTS_DIR
    / "retrieval_per_query_union_mw80_StdNP_N1000.tsv",
    "\u00b110 Da": RESULTS_DIR
    / "retrieval_per_query_union_mw10_StdNP_N1000.tsv",
    "\u00b15 Da": RESULTS_DIR
    / "retrieval_per_query_union_mw5_StdNP_N1000.tsv",
}

HA_UNION_FILES = {
    "\u00b11": RESULTS_DIR
    / "retrieval_per_query_union_heavy_atom1_StdNP_N1000.tsv",
    "\u00b12": RESULTS_DIR
    / "retrieval_per_query_union_heavy_atom2_StdNP_N1000.tsv",
    "\u00b13": RESULTS_DIR
    / "retrieval_per_query_union_heavy_atom3_StdNP_N1000.tsv",
    "\u00b16": RESULTS_DIR
    / "retrieval_per_query_union_heavy_atom6_StdNP_N1000.tsv",
    "\u00b18": RESULTS_DIR
    / "retrieval_per_query_union_heavy_atom8_StdNP_N1000.tsv",
}

mw_union_dfs = {}
for label, f in MW_UNION_FILES.items():
    if f.exists():
        mw_union_dfs[label] = pd.read_csv(f, sep="\t")
    else:
        print(f"MISSING (not yet written): {f}")

ha_union_dfs = {}
for label, f in HA_UNION_FILES.items():
    if f.exists():
        ha_union_dfs[label] = pd.read_csv(f, sep="\t")
    else:
        print(f"MISSING (not yet written): {f}")

print("MW-union files loaded:", list(mw_union_dfs.keys()))
print("HA-union files loaded:", list(ha_union_dfs.keys()))

## Panel A — recall vs. median candidate pool size (StdNP subset, n=1,168)

Recall = fraction of queries whose true molecule falls inside the candidate
window (an upper bound on achievable top-1 under autofail scoring). x-axis is
the median per-query pool size actually scanned, on a log scale. Union
traces are included automatically once their per-query files exist (see
cell above).

In [ ]:
def pool_and_recall(df):
    return df["n_candidates"].median(), df["true_mol_found"].mean()


def sorted_xy(points):
    xs = [p[0] for p in points]
    ys = [100 * p[1] for p in points]
    order = sorted(range(len(xs)), key=lambda i: xs[i])
    return [xs[i] for i in order], [ys[i] for i in order]


ri_points = [pool_and_recall(df) for df in ri_dfs.values()]
mw_points = [pool_and_recall(df) for df in mw_dfs.values()]
ha_points = [pool_and_recall(df) for df in ha_dfs.values()]

fig, ax = make_fig("square")

series = [
    (ri_points, "RI", palette[0], "o", "-"),
    (mw_points, "MW", palette[3], "s", "-"),
    (ha_points, "Heavy-atom count", palette[5], "^", "-"),
]
if mw_union_dfs:
    mw_union_points = [pool_and_recall(df) for df in mw_union_dfs.values()]
    series.append((mw_union_points, "RI\u222aMW union", palette[3], "s", "--"))
if ha_union_dfs:
    ha_union_points = [pool_and_recall(df) for df in ha_union_dfs.values()]
    series.append((ha_union_points, "RI\u222aHA union", palette[5], "^", "--"))

for points, label, color, marker, ls in series:
    xs, ys = sorted_xy(points)
    ax.plot(
        xs,
        ys,
        marker=marker,
        linestyle=ls,
        color=color,
        label=label,
        linewidth=1.5,
    )

ax.set_xscale("log")
ax.set_xlabel("Median candidate pool size")
ax.set_ylabel("Recall (%)")
ax.set_ylim(0, 105)
# ax.grid(True, linestyle="--", alpha=0.4, linewidth=0.5)
ax.legend(fontsize=7)

save_fig(fig, "recall_vs_pool_size_stdnp", OUTPUT_DIR)
plt.show()
plt.close(fig)

## Panel B — top-1 accuracy vs. median candidate pool size (StdNP subset, n=1,168)

Same x-axis and query population as Panel A. Dashed horizontal line =
unfiltered top-1 on this same StdNP subset (`retrieval_ablation_StdNP.json["all"]`).
Union traces included automatically once their per-query files exist.

In [ ]:
def pool_and_top1(df):
    return df["n_candidates"].median(), (
        df["rank_autofail_cosine"] <= 1
    ).mean()


ri_top1 = [pool_and_top1(df) for df in ri_dfs.values()]
mw_top1 = [pool_and_top1(df) for df in mw_dfs.values()]
ha_top1 = [pool_and_top1(df) for df in ha_dfs.values()]

fig, ax = make_fig("square")

series = [
    (ri_top1, "RI", palette[0], "o", "-"),
    (mw_top1, "MW", palette[3], "s", "-"),
    (ha_top1, "Heavy-atom count", palette[5], "^", "-"),
]
if mw_union_dfs:
    mw_union_top1 = [pool_and_top1(df) for df in mw_union_dfs.values()]
    series.append((mw_union_top1, "RI\u222aMW union", palette[3], "s", "--"))
if ha_union_dfs:
    ha_union_top1 = [pool_and_top1(df) for df in ha_union_dfs.values()]
    series.append((ha_union_top1, "RI\u222aHA union", palette[5], "^", "--"))

for points, label, color, marker, ls in series:
    xs, ys = sorted_xy(points)
    ax.plot(
        xs,
        ys,
        marker=marker,
        linestyle=ls,
        color=color,
        label=label,
        linewidth=1.5,
    )

ax.axhline(
    100 * unfiltered_top1_stdnp,
    color="gray",
    linestyle="--",
    linewidth=1.0,
    label="Unfiltered",
)

ax.set_xscale("log")
ax.set_xlabel("Median candidate pool size")
ax.set_ylabel("Top-1 accuracy (%)")
# ax.grid(True, linestyle="--", alpha=0.4, linewidth=0.5)
ax.legend(fontsize=7)

save_fig(fig, "top1_vs_pool_size_stdnp", OUTPUT_DIR)
plt.show()
plt.close(fig)

In [ ]:
def pool_and_top1(df):
    return df["n_candidates"].median(), (
        df["rank_autofail_cosine"] <= 1
    ).mean()


ri_top1 = [pool_and_top1(df) for df in ri_dfs.values()]
mw_top1 = [pool_and_top1(df) for df in mw_dfs.values()]
ha_top1 = [pool_and_top1(df) for df in ha_dfs.values()]

fig, ax = make_fig("square")

series = [
    (ri_top1, "RI", palette[0], "o", "-"),
    (mw_top1, "MW", palette[3], "s", "-"),
    (ha_top1, "Heavy-atom count", palette[5], "^", "-"),
]
# if mw_union_dfs:
#     mw_union_top1 = [pool_and_top1(df) for df in mw_union_dfs.values()]
#     series.append((mw_union_top1, "RI\u222aMW union", palette[3], "s", "--"))
# if ha_union_dfs:
#     ha_union_top1 = [pool_and_top1(df) for df in ha_union_dfs.values()]
#     series.append((ha_union_top1, "RI\u222aHA union", palette[5], "^", "--"))

for points, label, color, marker, ls in series:
    xs, ys = sorted_xy(points)
    ax.plot(
        xs,
        ys,
        marker=marker,
        linestyle=ls,
        color=color,
        label=label,
        linewidth=1.5,
    )

ax.axhline(
    100 * unfiltered_top1_stdnp,
    color="gray",
    linestyle="--",
    linewidth=1.0,
    label="Unfiltered",
)

ax.set_xscale("log")
ax.set_xlabel("Median candidate pool size")
ax.set_ylabel("Top-1 accuracy (%)")
# ax.grid(True, linestyle="--", alpha=0.4, linewidth=0.5)
ax.legend(fontsize=7)

save_fig(fig, "top1_vs_pool_size_stdnp_onlypure", OUTPUT_DIR)
plt.show()
plt.close(fig)

## Panel B2 — top-k accuracy vs. k, MW and heavy-atom count vs. unfiltered (full test set, n=27,647)

Same comparison as Panel C, but on the full 27,647-query test set (not the
StdNP subset) so the numbers are not inflated by the StdNP population's
smaller median MW/heavy-atom count (see caveat above). Best-performing
setting per descriptor by top-1 (Table `tab:si-mw-only` /
`tab:heavy-atom-retrieval`): MW at ±5 Da, HA at ±6 atoms. No RI line here
(RI has no full-test-set version).

In [ ]:
K_VALUES = list(range(1, 51))


def topk_curve(df, k_values):
    ranks = df["rank_autofail_cosine"]
    return [100 * (ranks <= k).mean() for k in k_values]


unfiltered_df_all = pd.read_csv(
    RESULTS_DIR / "retrieval_global_per_query.tsv", sep="\t"
)
mw5_all_df = pd.read_csv(
    RESULTS_DIR / "retrieval_per_query_mw5_all.tsv", sep="\t"
)
ha6_all_df = pd.read_csv(
    RESULTS_DIR / "retrieval_per_query_heavy_atom6_all.tsv", sep="\t"
)

fig, ax = make_fig("square")
for df, label, color in [
    (mw5_all_df, "MW, ±5 Da", palette[3]),
    (ha6_all_df, "Heavy-atom count, ±6", palette[5]),
]:
    ax.plot(
        K_VALUES,
        topk_curve(df, K_VALUES),
        color=color,
        label=label,
        linewidth=1.5,
    )
ax.plot(
    K_VALUES,
    topk_curve(unfiltered_df_all, K_VALUES),
    color="gray",
    linestyle="--",
    linewidth=1.0,
    label="Unfiltered",
)
ax.set_xlabel("k")
ax.set_ylabel("Top-k accuracy (%)")
# ax.grid(True, linestyle="--", alpha=0.4, linewidth=0.5)
ax.legend(fontsize=7)

save_fig(fig, "topk_vs_k_mw_ha_full_testset", OUTPUT_DIR)
plt.show()
plt.close(fig)

## Panel C — top-k accuracy vs. k, best setting per descriptor (StdNP subset, n=1,168)

Direct answer to "is there a distinct advantage to fewer candidates": each
descriptor's single best-performing window (by top-1, from the summary
table below) plotted as a full top-k curve, against the unfiltered
(full-library) baseline. Best settings: RI at N=10,000,000 (top-1=0.104,
the best non-"all" RI pool size); MW at ±5 Da (top-1=0.229); HA at ±1 atom
(top-1=0.263). A descriptor's curve lying above the unfiltered line at a
given k means filtering to that window is worthwhile at that k; below means
it is not.

In [ ]:
unfiltered_df = pd.read_csv(
    RESULTS_DIR / "retrieval_per_query_StdNP_all.tsv", sep="\t"
)

BEST_RI = ri_dfs[10_000_000]
BEST_MW = mw_dfs["±5 Da"]
BEST_HA = ha_dfs["±1"]

K_VALUES = list(range(1, 51))


def topk_curve(df, k_values):
    ranks = df["rank_autofail_cosine"]
    return [100 * (ranks <= k).mean() for k in k_values]


def plot_topk_panel(k_values, out_name):
    fig, ax = make_fig("square")
    for df, label, color in [
        (BEST_RI, "RI, N=10,000,000", palette[0]),
        (BEST_MW, "MW, ±5 Da", palette[3]),
        (BEST_HA, "Heavy-atom count, ±1", palette[5]),
    ]:
        ax.plot(
            k_values,
            topk_curve(df, k_values),
            color=color,
            label=label,
            linewidth=1.5,
        )
    ax.plot(
        k_values,
        topk_curve(unfiltered_df, k_values),
        color="gray",
        linestyle="--",
        linewidth=1.0,
        label="Unfiltered",
    )
    ax.set_xlabel("k")
    ax.set_ylabel("Top-k accuracy (%)")
    ax.legend(fontsize=7)
    save_fig(fig, out_name, OUTPUT_DIR)
    plt.show()
    plt.close(fig)


plot_topk_panel(K_VALUES, "topk_vs_k_best_setting_stdnp")

## Panel B10 — top-10 accuracy vs. median candidate pool size (StdNP subset, n=1,168)

Same as Panel B, using top-10 instead of top-1. Union traces included
automatically once their per-query files exist.

In [ ]:
def pool_and_topk(df, k):
    return df["n_candidates"].median(), (
        df["rank_autofail_cosine"] <= k
    ).mean()


K_FOR_PANEL_B10 = 10

ri_top10 = [pool_and_topk(df, K_FOR_PANEL_B10) for df in ri_dfs.values()]
mw_top10 = [pool_and_topk(df, K_FOR_PANEL_B10) for df in mw_dfs.values()]
ha_top10 = [pool_and_topk(df, K_FOR_PANEL_B10) for df in ha_dfs.values()]

unfiltered_top10_stdnp = ri_ladder["all"]["autofail"]["cosine"][
    "top_10_accuracy"
]

fig, ax = make_fig("square")

series = [
    (ri_top10, "RI", palette[0], "o", "-"),
    (mw_top10, "MW", palette[3], "s", "-"),
    (ha_top10, "Heavy-atom count", palette[5], "^", "-"),
]
if mw_union_dfs:
    mw_union_top10 = [
        pool_and_topk(df, K_FOR_PANEL_B10) for df in mw_union_dfs.values()
    ]
    series.append((mw_union_top10, "RI∪MW union", palette[3], "s", "--"))
if ha_union_dfs:
    ha_union_top10 = [
        pool_and_topk(df, K_FOR_PANEL_B10) for df in ha_union_dfs.values()
    ]
    series.append((ha_union_top10, "RI∪HA union", palette[5], "^", "--"))

for points, label, color, marker, ls in series:
    xs, ys = sorted_xy(points)
    ax.plot(
        xs,
        ys,
        marker=marker,
        linestyle=ls,
        color=color,
        label=label,
        linewidth=1.5,
    )

ax.axhline(
    100 * unfiltered_top10_stdnp,
    color="gray",
    linestyle="--",
    linewidth=1.0,
    label="Unfiltered",
)

ax.set_xscale("log")
ax.set_xlabel("Median candidate pool size")
ax.set_ylabel("Top-10 accuracy (%)")
# ax.grid(True, linestyle="--", alpha=0.4, linewidth=0.5)
ax.legend(fontsize=7)

save_fig(fig, "top10_vs_pool_size_stdnp", OUTPUT_DIR)
plt.show()
plt.close(fig)

## Numeric summary (StdNP subset, for caption / cross-check against suppinfo.tex)

In [ ]:
def rows_for(dfs, filter_name, window_prefix=""):
    out = []
    for label, df in dfs.items():
        pool, recall = pool_and_recall(df)
        _, top1 = pool_and_top1(df)
        out.append(
            {
                "filter": filter_name,
                "window": f"{window_prefix}{label}",
                "n": len(df),
                "median_cand": pool,
                "recall_pct": 100 * recall,
                "top1": top1,
            }
        )
    return out


rows = []
rows += rows_for(ri_dfs, "RI", "N=")
rows += rows_for(mw_dfs, "MW")
rows += rows_for(ha_dfs, "HA")
rows += rows_for(mw_union_dfs, "RI\u222aMW")
rows += rows_for(ha_union_dfs, "RI\u222aHA")

summary_df = pd.DataFrame(rows)
summary_df.to_csv(
    OUTPUT_DIR / "descriptor_diagnostic_summary_stdnp.csv", index=False
)
summary_df

## SI — full test set (n=27,647), no RI restriction

MW-alone and HA-alone on the full test set, matching Table 1 /
`tab:pubchem-global-nofilter` and `tab:si-recall-vs-pool`. RI has no
equivalent (requires a real RI value per query), so this SI panel is
MW/HA-only, without an RI trace.

In [ ]:
MW_FILES_ALL = {
    "\u00b180 Da": RESULTS_DIR / "retrieval_per_query_mw80_all.tsv",
    "[\u221210,+80] Da": RESULTS_DIR / "retrieval_per_query_mw10_80_all.tsv",
    "[\u221210,+10] Da": RESULTS_DIR / "retrieval_per_query_mw10_10_all.tsv",
    "\u00b15 Da": RESULTS_DIR / "retrieval_per_query_mw5_all.tsv",
}
HA_FILES_ALL = {
    "\u00b11": RESULTS_DIR / "retrieval_per_query_heavy_atom1_all.tsv",
    "\u00b12": RESULTS_DIR / "retrieval_per_query_heavy_atom2_all.tsv",
    "\u00b13": RESULTS_DIR / "retrieval_per_query_heavy_atom3_all.tsv",
    "\u00b16": RESULTS_DIR / "retrieval_per_query_heavy_atom6_all.tsv",
    "\u00b18": RESULTS_DIR / "retrieval_per_query_heavy_atom8_all.tsv",
}

mw_dfs_all = {
    label: pd.read_csv(f, sep="\t") for label, f in MW_FILES_ALL.items()
}
ha_dfs_all = {
    label: pd.read_csv(f, sep="\t") for label, f in HA_FILES_ALL.items()
}

with open(RESULTS_DIR / "retrieval_global_results.json") as fh:
    global_results = json.load(fh)
unfiltered_top1_all = global_results["all"]["autofail"]["cosine"][
    "top_1_accuracy"
]

fig, ax = make_fig("square")

for dfs, label, color, marker in [
    (mw_dfs_all, "MW (n=27,647)", palette[3], "s"),
    (ha_dfs_all, "Heavy-atom count (n=27,647)", palette[5], "^"),
]:
    points = [pool_and_top1(df) for df in dfs.values()]
    xs, ys = sorted_xy(points)
    ax.plot(xs, ys, marker=marker, color=color, label=label, linewidth=1.5)

ax.axhline(
    100 * unfiltered_top1_all,
    color="gray",
    linestyle="--",
    linewidth=1.0,
    label="Unfiltered (n=27,647)",
)

ax.set_xscale("log")
ax.set_xlabel("Median candidate pool size")
ax.set_ylabel("Top-1 accuracy (%)")
# ax.grid(True, linestyle="--", alpha=0.4, linewidth=0.5)
ax.legend(fontsize=7)

save_fig(fig, "top1_vs_pool_size_full_testset_SI", OUTPUT_DIR)
plt.show()
plt.close(fig)